# 00 — Setup, load, and config audit

**Project:** What Does the Artificial Hippocampus Store? · Kim & Nguyễn · mentor Gautam Siddharth Kashyap
**Stage:** Proposal Stage 1 (instrument the AHN forward pass) — the *first* half.

Gautam's instruction was "get one 3B checkpoint running … and make sure we can hook into
the AHN output/state correctly." This notebook does the first clause and, more
importantly, records the configuration values that silently determine what every later
measurement means.

### Why this notebook exists at all

Three config values change the interpretation of a retention curve and none of them is
visible in the output of a forward pass:

| value | why it matters |
|---|---|
| `num_attn_sinks` | the first *N* tokens are **never compressed** and stay losslessly visible to attention. Upstream eval uses 128. A needle inside that prefix was never evicted, so a "retention" measurement on it is measuring nothing. |
| `sliding_window` | AHN is inert below it. The 18 Aug pilot ran at 128 and 256 in two different notebooks; the proposal specifies 8064. |
| `use_ahn_router` | if true, memory enters through a learned sigmoid gate rather than a plain sum, and `o_proj(ahn_out)` becomes an upper bound on the contribution rather than the contribution. |

**Output:** `results/<run>/00_config_audit.json`. Attach it to anything you show Gautam.


In [1]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


ahn_interp loaded from /home/jupyter-dphs-da30/hannah/AHN
working directory pinned to /home/jupyter-dphs-da30/hannah/AHN


In [2]:
# --- experiment configuration ----------------------------------------------------
# Everything that changes what a number MEANS lives here and gets saved with the run.
CFG = dict(
    model_path      = "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
    cell            = "GatedDeltaNet",     # GatedDeltaNet | DeltaNet | Mamba2
    scale           = "3B",
    sliding_window  = 8064,                # proposal value; upstream eval uses 8064
    num_attn_sinks  = 128,                 # upstream eval default. NOT zero.
    attn_impl       = "flash_attention_2", # "eager" on T4/P100 (no Ampere -> no FA2)
    dtype           = "bfloat16",          # "float16" on T4/P100
    results_dir     = "results/run_3b_gdn",
)
ai.set_results_dir(CFG["results_dir"])
print(json.dumps(CFG, indent=2))


{
  "model_path": "/home/jupyter-dphs-4ca1/AHN/merged_ckpt/Qwen-2.5-Instruct-3B-AHN-GDN",
  "cell": "GatedDeltaNet",
  "scale": "3B",
  "sliding_window": 8064,
  "num_attn_sinks": 128,
  "attn_impl": "flash_attention_2",
  "dtype": "bfloat16",
  "results_dir": "results/run_3b_gdn"
}


In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f"VRAM free/total: {free/1e9:.1f} / {total/1e9:.1f} GB")
print("torch:", torch.__version__)

cap = torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0, 0)
if cap[0] < 8:
    print(f"\n! compute capability {cap} is pre-Ampere (T4=7.5, P100=6.0).")
    print("  FlashAttention-2 and bfloat16 are unavailable. Set:")
    print('    CFG["attn_impl"] = "eager";  CFG["dtype"] = "float16"')
    print("  This is Open Question 3 in the proposal — record whether it works.")


GPU: NVIDIA A100-SXM4-40GB
VRAM free/total: 8.6 / 42.4 GB
torch: 2.13.0+cu130


## Load the checkpoint

The proposal says to merge **in memory** rather than materialising ~150 GB of merged
directories. In practice `merge_weights.py` writes a directory once per checkpoint and
that directory is reused; at 3B that is ~6 GB, which is affordable. Decide once, record
the choice, and keep `merged_ckpt/` out of git (it already is).


In [4]:
bundle = ai.load_ahn_model(
    CFG["model_path"],
    dtype=getattr(torch, CFG["dtype"]),
    attn_implementation=CFG["attn_impl"],
    sliding_window=CFG["sliding_window"],
    num_attn_sinks=CFG["num_attn_sinks"],
)
print(json.dumps(bundle.summary(), indent=2))


/home/jupyter-dphs-da30/ahn-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ModuleNotFoundError: No module named 'ahn'

In [ ]:
audit = ai.audit_config(bundle)
audit["cfg"] = CFG
ai.save_json(audit, "00_config_audit.json")
print("\nsaved ->", os.path.join(CFG["results_dir"], "00_config_audit.json"))


## Read the combination site directly

Do not take my word for how the memory is combined. This prints the source of the
decoder layer's forward so you can see the two lines yourself:

```python
attn_output[:, -L:, :] = attn_output[:, -L:, :] + ahn_attn_output
hidden_states = self.self_attn.o_proj(attn_output)
```

Plain **addition, before `o_proj`**. That is the answer to Open Question 4 in the
proposal, and it means a forward hook on `layer.ahn` returns a vector in the
concatenated-head space, *not* the residual stream. On Qwen2.5-3B both are 2048-dim
(16 heads × 128), so decoding the hook output straight through the unembedding runs
without error and returns noise — which is what happened in the pilot.


In [ ]:
import inspect, re

L = bundle.ahn_layers[len(bundle.ahn_layers) // 2]
src = inspect.getsource(bundle.model.model.layers[L].forward)

for i, line in enumerate(src.splitlines()):
    if re.search(r"ahn_attn_output|o_proj|num_attn_sinks|in_ahn_seq_len", line):
        print(f"{i:4d} | {line}")


In [ ]:
# quick smoke test: does AHN activate, and where does the boundary fall?
tok = bundle.tokenizer
probe = ai.AHNProbe(bundle)

spec = ai.build_niah_prompt(tok, "Paris", bundle, eviction_distance=512)
print(json.dumps({k: v for k, v in spec.items() if k != "prompt"}, indent=2))

inputs = tok(spec["prompt"], return_tensors="pt").to(bundle.model.device)
cap0 = probe.run(inputs, layers=bundle.ahn_layers[:4])
print("\nAHN active:", cap0.ahn_active, "| layers captured:", cap0.captured_layers)
print("ahn_raw shape:", tuple(cap0.ahn_raw[bundle.ahn_layers[0]].shape))
print("o_t     shape:", tuple(cap0.o_t(bundle.ahn_layers[0], pos=None).shape))
ai.free_cuda()


### Gate for this notebook

- [ ] `ahn_will_activate` is `True` and layers were captured
- [ ] `needle_is_evicted` is `True` and `needle_in_sink_region` is `False`
- [ ] the audit's warning list is empty, or every warning is understood and written down
- [ ] `00_config_audit.json` saved

Then go to **01_instrumentation_gate.ipynb**.
